In [100]:
from typing import NamedTuple, TypedDict

import pandas as pd
import requests
from pathlib import Path

from narwhals import DataFrame

from tqdm.notebook import tqdm
import zipfile
import librosa
from IPython import display
from glob import glob
import numpy as np

In [101]:
np.random.seed(0)

In [102]:
# https://stackoverflow.com/a/62113293
def download(url, path: Path):
    resp = requests.get(url, stream=True)
    total = int(resp.headers.get('content-length', 0))
    with open(path, 'wb') as file, tqdm(
            desc=path.name,
            total=total,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
    ) as bar:
        for data in resp.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)

In [103]:
download_path = Path("data/dataset.zip")
if not (download_path.exists() and download_path.is_file()):
    download("https://zenodo.org/records/3371780/files/audio_mono-mic.zip?download=1", download_path)

In [104]:
guitar_set_dir = Path("data/GuitarSet")
if not (guitar_set_dir.exists() and guitar_set_dir.is_dir()):
    with zipfile.ZipFile(download_path, "r") as zip_ref:
        for member in tqdm(zip_ref.infolist(), desc="Extracting "):
            zip_ref.extract(member, guitar_set_dir)

In [105]:
example, sr = librosa.load(guitar_set_dir.joinpath("00_BN1-129-Eb_comp_mic.wav"))
display.Audio(data = example, rate=sr)

In [106]:
class Sample(TypedDict):
    audio: Path
    sample_start: int
    sample_end: int

In [107]:
sample_length = 10
dataset_list: list[Sample] = []
for file in glob(str(guitar_set_dir)+"/*"):
    sample, sr = librosa.load(guitar_set_dir.joinpath("00_BN1-129-Eb_comp_mic.wav"))
    length: float = len(sample) / sr
    sample_no = int(length // sample_length)
    remainder: float = length - sample_length * sample_no
    for i in range(sample_no):
        offset: float = np.random.random()*remainder
        remainder -= max(0, offset)
        start_index = int(offset*sr)
        end_index = min(int(offset*sr + sample_length*sr), len(sample))
        dataset_list.append(Sample(audio=Path(file), sample_start=start_index, sample_end=end_index))

dataset: DataFrame = pd.DataFrame(dataset_list)
del dataset_list
dataset


,audio,sample_start,sample_end
0,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,28128,248628
1,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,16538,237038
2,data/GuitarSet/05_BN3-119-G_solo_mic.wav,30893,251393
3,data/GuitarSet/05_BN3-119-G_solo_mic.wav,11093,231593
4,data/GuitarSet/04_Jazz1-200-B_solo_mic.wav,21713,242213
...,...,...,...
715,data/GuitarSet/01_Rock1-130-A_solo_mic.wav,8660,229160
716,data/GuitarSet/04_Rock3-117-Bb_comp_mic.wav,22191,242691
717,data/GuitarSet/04_Rock3-117-Bb_comp_mic.wav,15351,235851
718,data/GuitarSet/00_SS2-107-Ab_solo_mic.wav,17909,238409


In [108]:
def get_audio(df: DataFrame, i: int)->tuple[np.ndarray, int]:
    row = df.iloc[i]
    x, sr = librosa.load(row.audio)
    return x[row.sample_start:row.sample_end], sr

In [109]:
x, sr = get_audio(dataset, 0)
display.Audio(x, rate=sr)

In [110]:
x, sr = get_audio(dataset, 1)
display.Audio(x, rate=sr)